In [ ]:
import random
import requests
import time
import matplotlib.pyplot as plt

In [ ]:
NUM_NODES = 10
FANOUT = 2
ROUNDS = 15

# trạng thái node
nodes = [False] * NUM_NODES

# node 0 bắt đầu
nodes[0] = True

In [ ]:
def detect():
    tx = [random.random() for _ in range(30)]

    try:
        res = requests.post(
            "http://127.0.0.1:8000/predict",
            json={"transaction": tx}
        )
        return res.json()["fraud"]
    except:
        return False

In [ ]:
def gossip_push(nodes):
    new_nodes = nodes.copy()

    for i in range(len(nodes)):
        if nodes[i]:
            targets = random.sample(range(len(nodes)), FANOUT)

            for t in targets:
                if not nodes[t]:
                    new_nodes[t] = True
                    print(f"Node {i} PUSH → Node {t}")

    return new_nodes

In [ ]:
history = []

for r in range(ROUNDS):
    print(f"\n🔁 Round {r+1}")

    # AI detection tại node 0
    if detect():
        nodes[0] = True
        print("🚨 AI detected fraud at node 0")

    nodes = gossip_push(nodes)

    informed = sum(nodes)
    history.append(informed)

    print(f"📊 Informed: {informed}/{NUM_NODES}")

    if informed == NUM_NODES:
        break

    time.sleep(1)

In [ ]:
plt.plot(history, marker='o')
plt.title("Gossip PUSH (AI Agent)")
plt.xlabel("Round")
plt.ylabel("Informed Nodes")
plt.grid()
plt.show()